In [ ]:
# paths
import numpy as np
import pandas as pd
import sys

sys.path.append('helpers/pcca_fa')
import helpers.pcca_fa.pcca_fa_mdl as pf
from dual_pfc_funcs import getParams, load_dict

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.style.use('scifigs.mplstyle')

# parameters
SAVE_FIG = False
data_path = 'preprocessed_data/'
params = getParams()
subjects = params['subjects']
plot_sym = params['markers']
color_map = params['color_map']

In [ ]:
# Shuffle hemisphere labels to show that real correlations are greater than shuffled
pccafa = {}
pccafa_shuffle = {}
for sub in subjects:
    pccafa = {**pccafa, **load_dict(data_path + sub + '_pccafa_cv15dim.pkl')}
    pccafa_shuffle = {**pccafa_shuffle, **load_dict(data_path + sub + '_pccafa_cv15dim_shuffle.pkl')}
fnames = pccafa.keys()

df = pd.DataFrame(columns=['SessionName','SvW1','SvW2','SvL1','SvL2','Shuffle-SvW1','Shuffle-SvW2','Shuffle-SvL1','Shuffle-SvL2'])
for sess in fnames:
    mdl = pf.pcca_fa()
    mdl.set_params(pccafa[sess]['params'])
    true_psv = mdl.compute_psv()

    mdl = pf.pcca_fa()
    mdl.set_params(pccafa_shuffle[sess]['params'])
    shuffle_psv = mdl.compute_psv()
    
    df2 = {'SessionName':sess,
           'SvW1':true_psv['avg_psv_W_1'],'SvW2':true_psv['avg_psv_W_2'],'SvL1':true_psv['avg_psv_L_1'],'SvL2':true_psv['avg_psv_L_2'],
           'Shuffle-SvW1':shuffle_psv['avg_psv_W_1'],'Shuffle-SvW2':shuffle_psv['avg_psv_W_2'],'Shuffle-SvL1':shuffle_psv['avg_psv_L_1'],'Shuffle-SvL2':shuffle_psv['avg_psv_L_2']}
    df.loc[len(df)] = df2
# df

In [ ]:
# create graphics 
fig,ax = plt.subplots()

for sub,sym in zip(subjects,plot_sym):
    sub_prefix = sub[:2].title()
    filt = [sub_prefix in s for s in df['SessionName']]
    tmp_df = df[filt]

    psv_within = np.concatenate((tmp_df['Shuffle-SvL1'], tmp_df['Shuffle-SvL2']))
    psv_across = np.concatenate((tmp_df['Shuffle-SvW1'], tmp_df['Shuffle-SvW2']))
    ax.plot(psv_within,psv_across,marker=sym,ls='',color='k',label='Monkey {:s}'.format(sub_prefix),markersize=5,fillstyle='none')

max_x = 31
ax.plot([0,max_x],[0,max_x],'k--')

ax.set_xlabel('shuffled within-area %sv', color=color_map['within'])
ax.set_ylabel('shuffled across-area %sv', color=color_map['across'])

ax.set_xlim([0,max_x])
ax.set_ylim([0,max_x])
ax.set_xticks(np.arange(0,max_x,15))
ax.set_yticks(np.arange(0,max_x,15))

ax.set_aspect('equal')
ax.legend(loc='lower right')

if SAVE_FIG:
    pdf = PdfPages('figs/across_vs_within_shuffle.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

# difference histogram plot - insets
psv_within = np.concatenate((df['Shuffle-SvL1'], df['Shuffle-SvL2']))
psv_across = np.concatenate((df['Shuffle-SvW1'], df['Shuffle-SvW2']))

psv_diff = psv_within - psv_across

fig,ax = plt.subplots()

tmp = ax.plot(psv_diff.mean(),12,marker='v',color=[.5,.5,.5],ms=12)
ax.hist(psv_diff,bins=18,color=tmp[0].get_color())
ax.plot([0,0],ax.get_ylim(),'k--')
ax.set_title('%sv')
ax.set_xlabel(r'$\Delta$ %sv')

print('%sv diff - ', psv_diff.mean())

if SAVE_FIG:
    pdf = PdfPages('figs/across_vs_within_diffs_shuffle.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()